<a href="https://colab.research.google.com/github/suparuek2405/thai-bank-rag-qa/blob/main/notebooks/NB02_vector_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NB02 — Vector Database
# Thai Bank Financial Q&A System

In [ ]:
# Install dependencies

!pip install pymupdf langchain langchain-community chromadb sentence-transformers umap-learn matplotlib -q

import chromadb, sentence_transformers, fitz, langchain
print(f"ChromaDB:              {chromadb.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"PyMuPDF:               {fitz.__version__}")
print(f"LangChain:             {langchain.__version__}")

ChromaDB:              1.5.9
sentence-transformers: 5.5.1
PyMuPDF:               1.27.2.3
LangChain:             1.3.4


In [ ]:
# Mount Drive, clone/pull repo, set paths

from google.colab import drive
drive.mount('/content/drive')

import sys, os, subprocess

DRIVE_ROOT    = "/content/drive/MyDrive/Github experiment/thai-bank-rag-qa"
PROCESSED_DIR = f"{DRIVE_ROOT}/data/processed"
CHROMA_DIR    = "/content/chroma_db"   # local — fast read/write

REPO_DIR = "/content/thai-bank-rag-qa"
if not os.path.exists(REPO_DIR):
    get_ipython().system("git clone https://github.com/suparuek2405/thai-bank-rag-qa.git /content/thai-bank-rag-qa")
else:
    subprocess.run(["git", "pull", "--rebase", "origin", "main"], cwd=REPO_DIR)

sys.path.insert(0, REPO_DIR)

# Confirm chunk files are available
for f in sorted(os.listdir(PROCESSED_DIR)):
    size_mb = os.path.getsize(os.path.join(PROCESSED_DIR, f)) / 1_000_000
    print(f"  {f}  ({size_mb:.1f} MB)")

Mounted at /content/drive
Cloning into '/content/thai-bank-rag-qa'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 38 (delta 9), reused 29 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 24.64 KiB | 970.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.
  chunks_c1024_o100.json  (13.0 MB)
  chunks_c256_o50.json  (21.3 MB)
  chunks_c512_o100.json  (17.2 MB)


In [ ]:
# Load chunks (512/100 baseline config)

from src.loader import load_chunks

CHUNK_FILE = os.path.join(PROCESSED_DIR, "chunks_c512_o100.json")
chunks, config = load_chunks(CHUNK_FILE)

print(f"Loaded {len(chunks):,} chunks")
print(f"Config: {config}")

Loaded 27,813 chunks
Config: {'chunk_size': 512, 'overlap': 100, 'total_chunks': 27813, 'banks': ['BAY', 'BBL', 'CREDIT', 'KBANK', 'KKP', 'KTB', 'LHFG', 'SCBX', 'TISCO', 'TTB']}


In [ ]:
# Embed one chunk and inspect the output vector
# Before embedding everything, let's see what a single embedding looks like.
# This builds intuition for what "384-dimensional vector" means.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sample_texts = [
    "NPL ratio was 3.20% at the end of 2025.",
    "Non-performing loans stood at 3.20% of total loans in 2025.",
    "The bank opened 15 new branches in 2025."
]

embeddings = model.encode(sample_texts)

print(f"Embedding shape: {embeddings.shape}")   # (3, 384)
print(f"Each embedding has {embeddings.shape[1]} dimensions\n")

# Cosine similarity between the three texts
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim = cosine_similarity(embeddings)
labels = ["NPL sentence 1", "NPL sentence 2", "Unrelated sentence"]
print("Cosine similarity matrix:")
print(f"{'':20}", " ".join(f"{l:>20}" for l in labels))
for i, row in enumerate(sim):
    print(f"{labels[i]:20}", " ".join(f"{v:>20.4f}" for v in row))

In [ ]:
# Build ChromaDB vector store
# This is the slow cell — embeds all 27,813 chunks.
# On Colab CPU: ~10-15 minutes. On GPU: ~2-3 minutes.

from src.embedder import build_vectorstore

collection = build_vectorstore(
    chunks=chunks,
    chroma_dir=CHROMA_DIR,
    reset=False   # set True if you want to rebuild from scratch
)

print(f"\nCollection size: {collection.count():,} chunks")


In [ ]:
# Test single-bank retrieval
# Mode A: filter to one bank, then search.
# The metadata filter reduces search space from 27K → ~3.4K chunks.

from src.embedder import query, print_results

q = "What was KBANK's net interest margin in 2025?"
results = query(collection, question=q, bank_name="KBANK", top_k=10)
print_results(results, q)

# What to look for:
# - All results should be from KBANK (metadata filter working)
# - Top result should contain "NIM" or "net interest margin" with a % value
# - Score (cosine distance) should be < 0.5 for good matches


Query: "What was KBANK's net interest margin in 2025?"

[1] KBANK p.38  (score: 0.2208)
     'Net Interest Income \n KBank’s consolidated net interest income for 2025 was Baht 137,152 million, decreasing by Baht 10,852 million or 7.33 percent \nover-year. The decrease could be attributed to inte'

[2] KBANK p.41  (score: 0.2653)
     'KBank’s consolidated outstanding loans stood at Baht 2,476,647 million, decreasing by Baht 7,048 million \nor 0.28 percent, compared to Baht 2,483,695 million as of December 31, 2024.\nLoan Portfolio by'

[3] KBANK p.40  (score: 0.2688)
     '7,132 million or 7.10 percent from \nthe end of 2024, due mainly to KBank’s liquidity management.\n4.2 Financial Position\nInvestment in Securities\nAssets and Liabilities Structure\nDec. 31, 2025\nDec. 31,'

[4] KBANK p.40  (score: 0.2902)
     'aht 35,338 million or 20.53 percent from \nthe end of 2024 in line with KBank’s liquidity management. \n Equity (attributable to KBank) at the end of 2025 amounted to Baht 5

In [ ]:
# Quick check — does any KBANK chunk contain the NIM value?
nim_chunks = [c for c in chunks
              if c["bank_name"] == "KBANK"
              and "3.23" in c["text"]]
print(f"Chunks containing '3.23': {len(nim_chunks)}")
if nim_chunks:
    print(nim_chunks[0]["text"][:300])

Chunks containing '3.23': 5
As of or for the years ended December 31,
2025
2024 
(Restated)
2023
2022
2021
PERFORMANCE INDICATORS
Return on average assets (ROA)
1.11%
1.15%
0.99%
0.86%
0.98%
Return on average equity (ROE) 6
8.62%
9.13%
8.29%
7.38%
8.44%
Net interest margin (NIM)
3.23%
3.60%
3.66%
3.33%
3.21%
Cost to income rat


In [ ]:
# Test cross-bank retrieval
# Mode B: no filter, search all banks.
# Used for questions like "which bank had the highest NPL?"

q = "Which bank had the highest NPL ratio in 2025?"
results = query(collection, question=q, bank_name=None, top_k=20)
print_results(results, q)

# What to look for:
# - Results should come from multiple banks
# - Top results should contain NPL ratio values
# - Check if KKP (4.3%) appears — our eval answer says it had the highest NPL


Query: 'Which bank had the highest NPL ratio in 2025?'

[1] KTB p.100  (score: 0.245)
     'ronment of ongoing \neconomic uncertainties. NPL ratio was 2.90%, down from 2.99% at the end of 2024. The Bank maintained a high coverage ratio \nof 203.6%, compared with 188.6% in the previous year, in'

[2] TISCO p.23  (score: 0.2665)
     'on-performing loan (NPL) ratio stood at 2.84%, an increase from 2.78% at the end of the prior \nyear. In response, commercial banks continued to set aside provisions prudently, maintaining a high cover'

[3] KTB p.100  (score: 0.3035)
     '7\n- Equity holders of the Bank\n464,229\n11.8\n440,122\n11.8\n5.5\n- Non-controling interest\n22,793\n0.6\n20,549\n0.6\n10.9\nTotal liabilities and equity\n3,933,319\n100.0\n3,740,468\n100.0\n5.2\nLoans to customers \n('

[4] TTB p.100  (score: 0.3051)
     'dated basis, NPL ratio stood at 2.87% remained within the Bank’s guidance of below 2.9%.\n• \nOn bank-only basis, NPL ratio stood at 2.55%, compared to 2.32% as of 

In [ ]:
# Test retrieval for all val question types
# Quick sanity check: does retrieval work for each question category?

import json

with open(f"{REPO_DIR}/data/eval_questions.json") as f:
    eval_qs = json.load(f)

test_cases = [
    ("single_bank", eval_qs["val"][0]),   # V01 — BBL NPL
    ("cross_bank",  eval_qs["val"][6]),   # V07 — highest NPL
    ("trend",       eval_qs["val"][11]),  # V12 — KTB NIM trend
    ("regulatory",  eval_qs["val"][13]),  # V14 — BOT rate cut
]

for qtype, q_obj in test_cases:
    bank = q_obj["banks"][0] if len(q_obj["banks"]) == 1 else None
    results = query(collection, q_obj["question"], bank_name=bank, top_k=3)
    top = results[0]
    print(f"[{qtype.upper()}] {q_obj['id']}: {q_obj['question'][:60]}...")
    print(f"  Top result: {top['bank_name']} p.{top['page']}  score={top['score']}")
    print(f"  Preview: {top['text'][:120].strip()!r}")
    print()


[SINGLE_BANK] V01: What was BBL's NPL ratio at the end of 2025?...
  Top result: BBL p.93  score=0.478
  Preview: 'Classified Loans and Allowance for Expected Credit Losses\nAs of the end of December 2025, non-performing loan (Gross NPL'

[CROSS_BANK] V07: Which bank had the highest NPL ratio among all 10 banks in 2...
  Top result: KTB p.100  score=0.2746
  Preview: 'ronment of ongoing \neconomic uncertainties. NPL ratio was 2.90%, down from 2.99% at the end of 2024. The Bank maintained'

[TREND] V12: How did KTB's net interest margin (NIM) change from 2024 to ...
  Top result: KTB p.101  score=0.4009
  Preview: 'from January 1, 2016 until the capital buffer ratio of more than 2.5% is reached on January 1, 2019. Moreover, KTB \nwas'

[REGULATORY] V14: Which banks mentioned the BOT policy rate cut as a key facto...
  Top result: TISCO p.23  score=0.2644
  Preview: 'Part 1 Section 1: Group Structure and Business Operations \n \n \n20 \n \nIn 2025, the Bank of Thailand (BOT) reduced the 

In [ ]:
# Check if KTB NIM trend data exists in the chunks at all
nim_ktb = [c for c in chunks
           if c["bank_name"] == "KTB"
           and ("NIM" in c["text"] or "net interest margin" in c["text"].lower())]
print(f"KTB chunks mentioning NIM: {len(nim_ktb)}")
if nim_ktb:
    print(nim_ktb[0]["text"][:400])

KTB chunks mentioning NIM: 3
8,229
46,154 
 36,616 
33,698
21,588
Financial Ratios (%)
Return on Average Assets (ROA) (Equity Holders of the Bank)
1.26
1.24
1.01
0.94
0.63
Return on Average Equity (ROE) (Equity Holders of the Bank) 
10.67
10.94
9.40
9.15
6.14
Net Interest Margin (NIM) 
2.82
3.29
3.22
2.60
2.49
Cost to income ratio 
40.3
42.6
41.6
43.7
45.5
NPL Ratio
2.90
2.99
3.08
3.26
3.50
Coverage Ratio
203.6
188.6
181.3
17


In [ ]:
# UMAP visualization of embedding space
# Reduces 384-d embeddings to 2-d and plots by bank.
# If bank-specific clusters form, the embedding model captures bank identity.
# Uses a 2000-chunk sample to keep runtime reasonable.

import numpy as np
import matplotlib.pyplot as plt
import umap

BANKS = ["BBL", "KBANK", "KTB", "SCBX", "TTB", "TISCO", "BAY", "LHFG", "CREDIT", "KKP"]
COLORS = ["#e6194b","#3cb44b","#4363d8","#f58231","#911eb4",
          "#42d4f4","#f032e6","#bfef45","#fabebe","#469990"]

# Sample 200 chunks per bank for speed
np.random.seed(42)
sampled_chunks = []
for bank in BANKS:
    bank_chunks = [c for c in chunks if c["bank_name"] == bank]
    n = min(200, len(bank_chunks))
    sampled_chunks.extend(np.random.choice(bank_chunks, n, replace=False).tolist())

print(f"Sampled {len(sampled_chunks)} chunks for UMAP ({len(BANKS)} banks × ~200)")

# Embed the sample
texts = [c["text"] for c in sampled_chunks]
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(texts, batch_size=128, show_progress_bar=True)

# Reduce to 2-d
print("Running UMAP...")
reducer = umap.UMAP(n_components=2, metric="cosine", random_state=42, n_neighbors=15)
coords = reducer.fit_transform(embeddings)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
bank_labels = [c["bank_name"] for c in sampled_chunks]

for bank, color in zip(BANKS, COLORS):
    mask = [b == bank for b in bank_labels]
    x = coords[mask, 0]
    y = coords[mask, 1]
    ax.scatter(x, y, c=color, label=bank, alpha=0.5, s=10)

ax.legend(loc="upper right", markerscale=3)
ax.set_title("UMAP of chunk embeddings — colored by bank\n"
             "Clusters = embedding model captures bank-specific language", fontsize=13)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.tight_layout()

# Save to Drive
fig_dir = f"{DRIVE_ROOT}/results/figures"
os.makedirs(fig_dir, exist_ok=True)
fig_path = f"{fig_dir}/umap_embeddings.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved → {fig_path}")

In [ ]:
# Copy ChromaDB to Drive (persist across sessions)

import shutil

chroma_drive = f"{DRIVE_ROOT}/chroma_db"
if os.path.exists(chroma_drive):
    shutil.rmtree(chroma_drive)

shutil.copytree(CHROMA_DIR, chroma_drive)
size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, filenames in os.walk(chroma_drive)
    for f in filenames
) / 1_000_000
print(f"ChromaDB saved to Drive: {chroma_drive}  ({size_mb:.1f} MB)")

ChromaDB saved to Drive: /content/drive/MyDrive/Github experiment/thai-bank-rag-qa/chroma_db  (144.9 MB)
